# ARCHS4 saturation: coverage vs model rank

Coverage as a function of **K**, the number of latent variables, at several study
subsampling levels. The coverage analysis asks how much data CLAMP needs; this asks how
much model capacity it needs, and whether the two interact -- a small compendium may
saturate at a lower rank than a large one.

Subsampling is by study for the same reason as in `00_coverage`.

This notebook only plots; the ORA sweep runs in `scripts/archs4/ora_gpu.py` under
`workflow/rules/archs4_saturation.smk`. Figures are inline only.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(yaml)
    library(here)
})

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS <- unlist(cfg$MODEL_COLORS)
FRACTION_COLORS <- unlist(cfg$RNASEQ_FRACTION_COLORS)

theme_archs4 <- function(base_size = 11) {
    theme_classic(base_size = base_size) +
        theme(
            axis.text          = element_text(colour = "black"),
            axis.title         = element_text(colour = "black"),
            panel.grid.major.y = element_line(colour = "#E0E0E0", linewidth = 0.3),
            panel.grid.minor   = element_blank(),
            strip.background   = element_blank(),
            strip.text         = element_text(face = "bold")
        )
}

## Settings

In [ ]:
SATURATION_DIR <- here(snakemake@params[["saturation_dir"]])
FDR_LEVELS     <- as.numeric(snakemake@params[["fdr"]])
MODELS         <- c("CLAMPfull", "CLAMPbase")
LIBRARIES      <- c(bp = "GO:BP", msigdb = "MSigDB")

stopifnot(dir.exists(SATURATION_DIR))
cat("Saturation ORA root:", SATURATION_DIR, "\n")

## Load the sweep

Filenames carry both axes: `rs{level}_k{K}_seed{n}`. Everything else matches the coverage
notebook, including taking the pathway-count denominator from the `_meta.csv` sibling.

In [ ]:
load_point <- function(f) {
    m <- regmatches(basename(f),
                    regexec("^rs([0-9]+)_k([0-9]+)_seed([0-9]+)_.*_gpu_ora\\.csv$",
                            basename(f)))[[1]]
    if (length(m) < 4) return(NULL)
    meta_path <- sub("_(bp|msigdb)_gpu_ora\\.csv$", "_meta.csv", f)
    if (!file.exists(meta_path)) return(NULL)

    meta  <- fread(meta_path)
    terms <- fread(f)
    out <- data.table(
        level_pct = as.integer(m[2]),
        k         = as.integer(m[3]),
        seed      = as.integer(m[4]),
        n_samples = meta$n_samples[1],
        n_lvs     = meta$n_lvs[1],
        n_total   = meta$n_total_msigdb[1]
    )
    for (fdr in FDR_LEVELS) {
        out[[sprintf("coverage_%g", fdr)]] <-
            sum(terms$padj_min_across_lvs < fdr) / meta$n_total_msigdb[1]
    }
    out
}

saturation <- rbindlist(lapply(names(LIBRARIES), function(lib) {
    rbindlist(lapply(MODELS, function(model) {
        d <- file.path(SATURATION_DIR, paste0("gpu_ora_", lib), model)
        if (!dir.exists(d)) return(NULL)
        files <- list.files(d, pattern = "^rs[0-9]+_k[0-9]+_seed[0-9]+_.*_gpu_ora\\.csv$",
                            full.names = TRUE)
        res <- rbindlist(lapply(files, load_point))
        if (!nrow(res)) return(NULL)
        res[, `:=`(library = unname(LIBRARIES[[lib]]), model = model)]
    }), fill = TRUE)
}), fill = TRUE)

stopifnot(nrow(saturation) > 0)
saturation[, level := paste0(level_pct, "%")]
setorder(saturation, library, model, level_pct, k, seed)
saturation[, .(n_points = .N, k_values = uniqueN(k)), by = .(library, model)]

## Coverage against rank, one curve per subsampling level

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

plot_saturation <- function(fdr, model_name) {
    col <- sprintf("coverage_%g", fdr)
    d <- saturation[model == model_name]
    if (!nrow(d)) return(invisible(NULL))
    ggplot(d, aes(x = k, y = 100 * .data[[col]], colour = level, group = level)) +
        stat_summary(fun = mean, geom = "line", linewidth = 0.8) +
        stat_summary(fun = mean, geom = "point", size = 1.8) +
        facet_wrap(~ library, scales = "free_y") +
        scale_x_log10() +
        labs(
            x = "Latent variables (K, log scale)",
            y = sprintf("Pathways recovered (%%, FDR < %s)", fdr),
            colour = "Studies used",
            title = model_name
        ) +
        theme_archs4() +
        theme(legend.position = "top")
}

plot_saturation(FDR_LEVELS[1], "CLAMPfull")

In [ ]:
plot_saturation(FDR_LEVELS[1], "CLAMPbase")

## Do bigger compendia need a bigger K?

Coverage per latent variable. If the curves peak at the same K regardless of how many
studies went in, rank and data size are separable choices.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 5)
col <- sprintf("coverage_%g", FDR_LEVELS[1])
per_lv <- saturation[model == "CLAMPfull"][
    , .(coverage_per_lv = 100 * mean(.SD[[1]]) / k), by = .(library, level, k),
    .SDcols = col]

ggplot(per_lv, aes(x = k, y = coverage_per_lv, colour = level, group = level)) +
    geom_line(linewidth = 0.8) +
    geom_point(size = 1.6) +
    facet_wrap(~ library, scales = "free_y") +
    scale_x_log10() +
    labs(x = "Latent variables (K, log scale)",
         y = "Pathways recovered per LV (%)",
         colour = "Studies used") +
    theme_archs4() +
    theme(legend.position = "top")

## Panel-ready tables

In [ ]:
fwrite(saturation, snakemake@output[["saturation_long"]])

panel_ready <- saturation[, .(
    mean_coverage = mean(.SD[[1]]),
    sd_coverage   = sd(.SD[[1]]),
    n_seeds       = .N
), by = .(library, model, level_pct, k), .SDcols = sprintf("coverage_%g", FDR_LEVELS[1])]
setorder(panel_ready, library, model, level_pct, k)
fwrite(panel_ready, snakemake@output[["panel_ready"]])
head(panel_ready, 12)